# IV. Thống kê dữ liệu

Phần này thực hiện phân tích chi tiết về bộ dữ liệu và trả lời các câu hỏi ở phần **III**

## 4.1 Import Libraries và Cấu hình

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway
import warnings
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor, 
    GradientBoostingRegressor, 
    ExtraTreesRegressor
)
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
sns.set_style('whitegrid')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 4.2 Load Data

In [ ]:
df = pd.read_csv('../dataset/track_data_final.csv')
print(f'Loaded {len(df):,} tracks')
df.head(3)

## 4.3 Data Preprocessing

### Workflow Tổng Quan

Preprocessing được thực hiện theo các bước sau:

1. **Convert datetime**: Chuyển `album_release_date` sang datetime để extract year
2. **Tạo biến phái sinh**: Thêm `duration_min`, `year` từ dữ liệu gốc
3. **Xử lý missing values**: Loại bỏ records với missing values trong các cột quan trọng
4. **Chuyển đổi kiểu dữ liệu**: Convert các cột số về đúng type (int64)

Mỗi câu hỏi sẽ có preprocessing riêng để tạo subset data phù hợp.

### Bước 1: Convert datetime

In [ ]:
df['album_release_date'] = pd.to_datetime(df['album_release_date'], format='mixed', errors='coerce')

### Bước 2: Tạo biến phái sinh

In [ ]:
df['duration_min'] = df['track_duration_ms'] / 60000
df['year'] = df['album_release_date'].dt.year

### Bước 3: Xử lý missing values

In [ ]:
df = df.dropna(subset=['year', 'artist_popularity', 'artist_followers'])
df.info()

### Bước 4: Chuyển đổi kiểu dữ liệu

In [ ]:
df['artist_popularity'] = df['artist_popularity'].astype('int64')
df['artist_followers'] = df['artist_followers'].astype('int64')
df['year'] = df['year'].astype('int64')

---

## 4.4 Câu hỏi 1: So sánh loại album

### Question

Loại album nào (`album_type`: album, compilation, single) có độ phổ biến cao hơn?

### A. Preprocessing

Kiểm tra phân bố các loại album_type và tính statistics cơ bản cho từng nhóm.

1. **Kiểm tra unique values**: Xác định các loại album_type có trong dataset
2. **Aggregate statistics**: Tính mean, median, count của track_popularity cho mỗi album_type
3. **Create comparison dataset**: Chuẩn bị data cho visualization và analysis

In [ ]:
q1_album_types = df['album_type'].value_counts()
q1_album_types

In [ ]:
q1_stats = df.groupby('album_type').agg({
    'track_popularity': ['mean', 'median', 'std', 'count'],
    'artist_popularity': ['mean', 'median']
}).round(2)

q1_stats.columns = ['_'.join(col) for col in q1_stats.columns]
q1_stats



### B. Analysis

**Phương pháp phân tích:**

So sánh track popularity giữa 3 loại album_type (album, single, compilation) để xác định format nào có popularity cao hơn:

1. **Grouped comparison**: So sánh mean và median track_popularity giữa các album types để identify differences
2. **Distribution analysis**: Box plots để visualize full distribution và identify outliers
3. **Statistical test**: ANOVA test để check xem differences có statistically significant không

**Expected outputs:**
- Bar chart: Mean track popularity by album_type
- Box plot: Popularity distribution comparison
- Statistics: ANOVA results và pairwise comparisons

**Rationale:** Mean comparison cho overall trend, box plot reveals quartiles và outliers, ANOVA test provides statistical evidence về whether observed differences are real.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

q1_mean_pop = q1_stats['track_popularity_mean'].sort_values(ascending=False)
colors_q1 = ['#6FBEE0', '#DAAF65', '#42AB84']

axes[0].bar(range(len(q1_mean_pop)), q1_mean_pop.values, 
            color=colors_q1, alpha=0.8, edgecolor='black')
axes[0].set_xticks(range(len(q1_mean_pop)))
axes[0].set_xticklabels(q1_mean_pop.index, rotation=15, ha='right')
axes[0].set_ylabel('Mean Track Popularity', fontweight='bold')
axes[0].set_title('Mean Track Popularity by Album Type')
axes[0].grid(axis='y', alpha=0.3)

for i, val in enumerate(q1_mean_pop.values):
    axes[0].text(i, val + 1, f'{val:.1f}', ha='center', fontweight='bold')

q1_album_data = df['album_type'].unique()
box_data = [df[df['album_type'] == atype]['track_popularity'] for atype in q1_album_data]
bp = axes[1].boxplot(box_data, labels=q1_album_data, patch_artist=True, widths=0.6)

for patch, color in zip(bp['boxes'], colors_q1):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].set_ylabel('Track Popularity', fontweight='bold')
axes[1].set_title('Track Popularity Distribution by Album Type')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
q1_album = df[df['album_type'] == 'album']['track_popularity']
q1_single = df[df['album_type'] == 'single']['track_popularity']
q1_compilation = df[df['album_type'] == 'compilation']['track_popularity']

f_stat, p_value = f_oneway(q1_album, q1_single, q1_compilation)

print(f'ANOVA F-statistic: {f_stat:.4f}')
print(f'P-value: {p_value:.4e}')
print(f'\nStatistical significance: {'Yes' if p_value < 0.05 else 'No'} (α=0.05)')


### C. Results & Interpretation

**Câu trả lời:** Albums có độ phổ biến cao nhất với mean popularity 55.5, trong khi singles ở 46.1 và compilations thấp nhất 40.5.

**Phát hiện chính:**

Full albums outperform singles với margin 9.4 điểm (55.5 vs 46.1), và vượt compilations tới 15 điểm. ANOVA test cho p-value < 0.001, chứng tỏ differences này highly significant và không phải do chance.

Median popularity cũng theo pattern tương tự: album (61) > single (44) > compilation (44). Box plots reveal rằng albums có distribution concentrated ở higher range với Q1=44 và Q3=73, trong khi singles và compilations có lower quartiles và more spread.

Compilations có popularity thấp nhất mặc dù thường gather "best of" tracks - có thể do listener preference cho cohesive album experiences hay do compilations bị coi là re-releases without new content.

**Ý nghĩa thực tiễn:**

Artists nên consider full album releases thay vì chỉ singles khi có enough material - data cho thấy full albums achieve higher popularity despite longer listening commitment. Labels investing trong album production có thể expect better popularity returns so với single-only strategies.

Tuy nhiên, cần note rằng singles có count cao hơn (2,248 tracks) so với albums trong recent years - xu hướng single releases có thể do cost considerations và streaming economics, không chỉ popularity potential.

**Limitations:**

Analysis không account for promotional budgets - albums thường có larger marketing spend so với singles. Cũng không phân biệt debut albums vs established artist albums. Dataset có thể bias towards successful albums (survivorship).


---

## 4.5 Câu hỏi 2: Explicit content và popularity

### Question

Những bài hát có explicit content (`explicit` = True) có độ phổ biến cao hơn hay thấp hơn so với những bài không explicit?

### A. Preprocessing

Kiểm tra phân bố explicit vs non-explicit tracks và tính statistics cho comparison.

1. **Check distribution**: Xác định tỷ lệ explicit vs non-explicit trong dataset
2. **Compute statistics**: Tính mean, median, std của track_popularity cho cả 2 groups
3. **Prepare comparison data**: Tạo datasets cho statistical testing và visualization

In [ ]:
q2_explicit_dist = df['explicit'].value_counts()
print('Distribution:')
print(q2_explicit_dist)
print(f'\nExplicit: {q2_explicit_dist[True] / len(df) * 100:.2f}%')
print(f'Non-Explicit: {q2_explicit_dist[False] / len(df) * 100:.2f}%')

In [ ]:
q2_stats = df.groupby('explicit').agg({
    'track_popularity': ['mean', 'median', 'std', 'count'],
    'artist_popularity': ['mean', 'median']
}).round(2)

q2_stats.columns = ['_'.join(col) for col in q2_stats.columns]
q2_stats


### B. Analysis

**Phương pháp phân tích:**

So sánh track popularity giữa explicit và non-explicit content để xác định content type nào có popularity cao hơn:

1. **Grouped comparison**: So sánh mean và median track_popularity giữa explicit vs non-explicit
2. **Distribution visualization**: Box plots và histograms để compare full distributions
3. **Statistical test**: T-test để check xem difference có statistically significant không

**Expected outputs:**
- Bar chart: Mean popularity comparison
- Box plot: Distribution comparison
- Histogram overlay: Popularity distributions cho cả 2 groups
- T-test results: Statistical significance

**Rationale:** T-test appropriate cho comparing 2 groups. Box plots reveal quartiles structure, histograms show shape của distributions để identify bimodality hoặc skewness.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

q2_mean_pop = q2_stats['track_popularity_mean'].sort_values(ascending=False)
colors_q2 = ['#519CBA', '#68F1BF']
labels_q2 = ['Explicit', 'Non-Explicit']

axes[0].bar(range(len(q2_mean_pop)), q2_mean_pop.values, 
            color=colors_q2, alpha=0.8, edgecolor='black')
axes[0].set_xticks(range(len(q2_mean_pop)))
axes[0].set_xticklabels(labels_q2, rotation=0)
axes[0].set_ylabel('Mean Track Popularity', fontweight='bold')
axes[0].set_title('Mean Track Popularity by Content Type')
axes[0].grid(axis='y', alpha=0.3)

for i, val in enumerate(q2_mean_pop.values):
    axes[0].text(i, val + 1, f'{val:.1f}', ha='center', fontweight='bold')

q2_explicit = df[df['explicit'] == True]['track_popularity']
q2_non_explicit = df[df['explicit'] == False]['track_popularity']

box_data = [q2_explicit, q2_non_explicit]
bp = axes[1].boxplot(box_data, labels=labels_q2, patch_artist=True, widths=0.6)

for patch, color in zip(bp['boxes'], colors_q2):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].set_ylabel('Track Popularity', fontweight='bold')
axes[1].set_title('Track Popularity Distribution by Content Type')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(q2_non_explicit, bins=30, alpha=0.6, label='Non-Explicit', 
        color='#68F1BF', edgecolor='black')
ax.hist(q2_explicit, bins=30, alpha=0.6, label='Explicit', 
        color='#519CBA', edgecolor='black')

ax.axvline(q2_non_explicit.mean(), color='#68F1BF', linestyle='--', 
           linewidth=2, label=f'Non-Explicit mean: {q2_non_explicit.mean():.1f}')
ax.axvline(q2_explicit.mean(), color='#519CBA', linestyle='--', 
           linewidth=2, label=f'Explicit mean: {q2_explicit.mean():.1f}')

ax.set_xlabel('Track Popularity', fontweight='bold')
ax.set_ylabel('Frequency', fontweight='bold')
ax.set_title('Popularity Distribution Overlay')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
t_stat, p_value = stats.ttest_ind(q2_explicit, q2_non_explicit)

print(f'T-statistic: {t_stat:.4f}')
print(f'P-value: {p_value:.4e}')
print(f'\nMean difference: {q2_explicit.mean() - q2_non_explicit.mean():.2f}')
print(f'Statistical significance: {'Yes' if p_value < 0.05 else 'No'} (α=0.05)')


### C. Results & Interpretation

**Câu trả lời:** Explicit tracks có độ phổ biến cao hơn đáng kể so với non-explicit tracks.

**Phát hiện chính:**

Explicit content đạt mean popularity 57.5 so với non-explicit 50.5 - chênh lệch 7 điểm. T-test cho p-value < 0.001, highly significant, chứng tỏ difference không phải random.

Median cũng cho thấy pattern tương tự: explicit (64) vượt non-explicit (55) 9 điểm. Box plots reveal rằng explicit tracks có distribution concentrated ở higher range với Q3=73 so với Q3=70 của non-explicit.

Distribution histograms cho thấy explicit tracks có peak ở 60-70 range, trong khi non-explicit có more spread với significant mass ở 30-50 range. Cả 2 distributions đều right-skewed với long tail về low popularity.

Artist popularity cũng cao hơn cho explicit content (75.8 vs 68.0), suggesting established artists với large fanbases có more freedom để release explicit content without fear of losing audience.

**Ý nghĩa thực tiễn:**

Data cho thấy explicit content không làm giảm popularity như nhiều người lo ngại - trái lại còn associate với higher popularity. Điều này có thể do established artists (đã có fanbase) thoải mái hơn với explicit lyrics, hoặc do explicit content resonate more với core streaming demographics (younger audiences).

Artists không cần self-censor để chase popularity - authentic expression qua explicit lyrics không penalty về mặt commercial success. Labels và platforms cũng nên reassess concerns về explicit content limiting reach.

**Limitations:**

Correlation không mean causation - explicit content không trực tiếp cause higher popularity. Có thể confounding factors như genre (hip-hop thường explicit và popular), artist status, hoặc target demographics. Analysis không control cho những variables này.


---

## 4.6 Câu hỏi 3: Golden duration analysis

### Question

Có tồn tại một khoảng thời lượng golden duration nào cho track mà trong đó những bài hát có độ phổ biến trung bình cao nhất không? Nếu có, khoảng này là bao nhiêu phút?

### A. Preprocessing

Sử dụng data đã được clean ở section 3.3. Không cần preprocessing thêm.

### B. Analysis

Phân tích duration theo 2 cấp độ:

1. **Coarse-grained analysis**: Chia duration thành 5 groups (<2p, 2-3p, 3-4p, 4-5p, >5p) để xem trend tổng quát
2. **Fine-grained analysis**: Chia duration thành bins 0.5 phút để tìm golden duration chính xác
3. **Visualizations**: Bar chart cho groups, scatter + trend line cho fine-grained analysis

In [ ]:
q3_df = df[['duration_min', 'track_popularity']].copy()

q3_df['duration_group'] = pd.cut(q3_df['duration_min'], 
                                   bins=[0, 2, 3, 4, 5, 100],
                                   labels=['<2p', '2-3p', '3-4p', '4-5p', '>5p'])

q3_group_stats = q3_df.groupby('duration_group')['track_popularity'].agg(['mean', 'median', 'count'])
q3_group_stats

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x_pos = np.arange(len(q3_group_stats))
bars = ax.bar(x_pos, q3_group_stats['mean'], 
               color=['#FF6B6B', '#FFA07A', '#4ECDC4', '#45B7D1', '#95A5A6'], 
               alpha=0.8, edgecolor='black')

for i, (bar, val) in enumerate(zip(bars, q3_group_stats['mean'])):
    ax.text(bar.get_x() + bar.get_width()/2., val + 1,
            f'{val:.1f}', ha='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Duration Group', fontweight='bold')
ax.set_ylabel('Mean Popularity', fontweight='bold')
ax.set_title('Popularity by Duration Groups')
ax.set_xticks(x_pos)
ax.set_xticklabels(q3_group_stats.index)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
bins = np.arange(0, 8, 0.5)
q3_df['duration_bin'] = pd.cut(q3_df['duration_min'], bins=bins)

q3_detailed_stats = q3_df.groupby('duration_bin')['track_popularity'].agg(['mean', 'count'])
q3_detailed_stats = q3_detailed_stats[q3_detailed_stats['count'] >= 30].sort_values('mean', ascending=False)

q3_detailed_stats.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

valid_stats = q3_detailed_stats[q3_detailed_stats.index.map(lambda x: x.right <= 7)]
bin_mids = [interval.mid for interval in valid_stats.index]

ax.scatter(bin_mids, valid_stats['mean'], s=valid_stats['count']/2, 
           c=valid_stats['mean'], cmap='RdYlGn', alpha=0.6, edgecolors='black')

z = np.polyfit(bin_mids, valid_stats['mean'], 3)
p = np.poly1d(z)
x_smooth = np.linspace(min(bin_mids), max(bin_mids), 200)
ax.plot(x_smooth, p(x_smooth), 'r--', linewidth=2, label='Trend')

ax.axvspan(3.5, 4.5, alpha=0.2, color='gold', label='Golden Zone (3.5-4.5p)')

ax.set_xlabel('Duration (phút)', fontweight='bold')
ax.set_ylabel('Mean Popularity', fontweight='bold')
ax.set_title('Duration vs Popularity (Fine-grained Analysis)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### C. Results & Interpretation

**Câu trả lời:** Có tồn tại golden duration - khoảng **3.5 - 4.5 phút** có mean popularity cao nhất (~55.6-55.9).

**Phát hiện chính:**

Popularity tăng dần từ bài hát ngắn (<2p: 38.2) đến medium (3-4p: 54.8) và peak ở 4-5p (55.3), sau đó giảm xuống khi bài hát quá dài (>5p: 50.3). Fine-grained analysis cho thấy peak chính xác ở 4.0-4.5 phút với popularity 55.9.

**Ý nghĩa thực tiễn:**

Producers và songwriters nên target duration 3.5-4.5 phút cho optimal popularity. Bài hát dưới 2 phút có xu hướng bị coi là thiếu substance (popularity thấp hơn 17 điểm so với golden zone), trong khi bài trên 5 phút có thể khiến listeners skip hoặc lose interest (giảm 5 điểm popularity).

**Limitations:**

Phân tích này không phân biệt genre - có thể một số genres (EDM, classical) có optimal duration khác. Dataset tập trung vào mainstream pop/hip-hop nên findings có thể không generalize cho tất cả music styles.

---

## 4.7 Câu hỏi 4: Số lượng bài hát và độ nổi tiếng nghệ sĩ

### Question

Nghệ sĩ phát hành nhiều bài hát hơn thì có đạt được độ nổi tiếng trung bình cao hơn so với nghệ sĩ phát hành ít bài hơn không?

### A. Preprocessing

Aggregate data theo artist để đếm số bài hát và tính mean popularity cho mỗi nghệ sĩ.

In [ ]:
q4_artist_stats = df.groupby('artist_name').agg({
    'track_id': 'count',
    'artist_popularity': 'first',
    'artist_followers': 'first'
}).rename(columns={'track_id': 'track_count'})

q4_artist_stats['output_strategy'] = pd.cut(q4_artist_stats['track_count'], 
                                              bins=[0, 20, 50, 100, 1000],
                                              labels=['Low (1-20)', 'Medium (21-50)', 'High (51-100)', 'Very High (>100)'])

q4_artist_stats.head()

### B. Analysis

**Phương pháp phân tích:**

Để trả lời câu hỏi này, nhóm sử dụng 2 phương pháp:

1. **So sánh theo nhóm**: Chia nghệ sĩ thành 4 nhóm theo chiến lược sản xuất (Low: 1-20 tracks, Medium: 21-50, High: 51-100, Very High: >100) để so sánh mean/median artist popularity giữa các nhóm
2. **Phân tích tương quan**: Scatter plot giữa số lượng track và artist popularity với đường xu hướng để thấy mối quan hệ

**Expected outputs:**
- Statistics table: Mean, median, std của artist popularity và followers cho mỗi output strategy group
- Visualization 1: Bar chart comparing mean artist popularity across groups
- Visualization 2: Scatter plot với trend line showing relationship giữa số lượng tracks và artist popularity

**Rationale:** Grouped analysis giúp thấy clear differences giữa strategies, trong khi scatter plot reveals detailed relationship pattern (linear, quadratic, hoặc plateau).

In [ ]:
q4_strategy_stats = q4_artist_stats.groupby('output_strategy').agg({
    'artist_popularity': ['mean', 'median', 'std'],
    'artist_followers': ['mean', 'median'],
    'track_count': 'count'
}).round(2)

q4_strategy_stats.columns = ['_'.join(col) for col in q4_strategy_stats.columns]
q4_strategy_stats

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].bar(range(len(q4_strategy_stats)), q4_strategy_stats['artist_popularity_mean'],
            color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'], alpha=0.8, edgecolor='black')
axes[0].set_xticks(range(len(q4_strategy_stats)))
axes[0].set_xticklabels(q4_strategy_stats.index, rotation=20, ha='right')
axes[0].set_ylabel('Mean Artist Popularity', fontweight='bold')
axes[0].set_title('Artist Popularity by Output Strategy')
axes[0].grid(axis='y', alpha=0.3)

for i, val in enumerate(q4_strategy_stats['artist_popularity_mean']):
    axes[0].text(i, val + 2, f'{val:.1f}', ha='center', fontweight='bold')

q4_plot_df = q4_artist_stats[q4_artist_stats['track_count'] <= 200]
axes[1].scatter(q4_plot_df['track_count'], q4_plot_df['artist_popularity'], 
                alpha=0.3, s=30, c=q4_plot_df['artist_popularity'], cmap='viridis')

z = np.polyfit(q4_plot_df['track_count'], q4_plot_df['artist_popularity'], 2)
p = np.poly1d(z)
x_line = np.linspace(q4_plot_df['track_count'].min(), q4_plot_df['track_count'].max(), 100)
axes[1].plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')

axes[1].set_xlabel('Number of Tracks', fontweight='bold')
axes[1].set_ylabel('Artist Popularity', fontweight='bold')
axes[1].set_title('Tracks Count vs Artist Popularity')
axes[1].legend()
axes[1].grid(alpha=0.3)

strategy_order = ['Low (1-20)', 'Medium (21-50)', 'High (51-100)', 'Very High (>100)']
sns.violinplot(data=q4_artist_stats, x='output_strategy', y='artist_popularity', order=strategy_order,
               palette=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'], ax=axes[2], inner='quartile')

# Vẽ thêm mean và median cho mỗi strategy
for i, strategy in enumerate(strategy_order):
    strategy_data = q4_artist_stats[q4_artist_stats['output_strategy'] == strategy]['artist_popularity']
    mean_val = strategy_data.mean()
    median_val = strategy_data.median()
    
    # Vẽ mean (hình vuông đỏ)
    axes[2].scatter(i, mean_val, color='red', s=100, marker='s', zorder=3, 
                    edgecolors='white', linewidths=2, label='Mean' if i == 0 else '')
    # Vẽ median (hình tròn xanh)
    axes[2].scatter(i, median_val, color='blue', s=100, marker='o', zorder=3, 
                    edgecolors='white', linewidths=2, label='Median' if i == 0 else '')

axes[2].set_title('Artist Popularity Distribution by Strategy', fontweight='bold')
axes[2].set_xlabel('Output Strategy', fontweight='bold')
axes[2].set_ylabel('Artist Popularity', fontweight='bold')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=20, ha='right')
axes[2].grid(alpha=0.3, axis='y')
axes[2].legend(loc='lower right', framealpha=0.9)

plt.tight_layout()
plt.show()

### C. Results & Interpretation

**Câu trả lời:** Có, nghệ sĩ phát hành nhiều bài hát hơn đạt được artist popularity cao hơn đáng kể.

**Phát hiện chính:**

Artist popularity tăng mạnh theo số lượng tracks: Low output (1-20 tracks) có mean popularity 52.9, trong khi Very High output (>100 tracks) đạt 94.3 - tăng gần gấp đôi. Medium output (21-50 tracks) đã có popularity 79.8, cho thấy consistent output là yếu tố quan trọng để build artist fame.

Artist followers cũng tăng theo cùng pattern: Low output có mean 2.4M followers, Very High output có 121.9M - tăng hơn 50 lần, chứng tỏ prolific output strategy không chỉ tăng popularity score mà còn build được massive fanbase.

Scatter plot cho thấy relationship không hoàn toàn linear - có diminishing returns sau ~100 tracks, và một số artists với low output vẫn có high popularity (có thể do quality hoặc viral hits).

**Ý nghĩa thực tiễn:**

Emerging artists nên aim for consistent releases thay vì perfecting một vài tracks - data cho thấy crossing từ Low sang Medium output (21-50 tracks) mang lại popularity jump lớn nhất (+26.9 points). Labels nên invest vào artists có capacity để maintain steady output, vì đây là strong predictor của long-term success.

**Limitations:**

Analysis này không account for time - artist có 100 tracks trong 1 năm khác với 100 tracks trong 10 năm. Cũng không phân biệt quality - popularity cao có thể do quantity hoặc do có hits trong catalog. Dataset bias towards successful artists (survivors) nên không thấy được unsuccessful prolific artists đã quit.


---

## 4.8 Câu hỏi 5: Classic songs vs Recent songs

### Question

Những bài hát phát hành từ 2009-2015 (classic songs) có còn giữ được độ phổ biến cao so với những bài hát phát hành từ 2020-2025 (recent songs) không?

### A. Preprocessing

Lọc data theo 2 time periods và loại bỏ tracks với popularity = 0 (dead songs).

In [ ]:
q5_classic = df[(df['year'] >= 2009) & (df['year'] <= 2015)].copy()
q5_recent = df[(df['year'] >= 2020) & (df['year'] <= 2025)].copy()

q5_classic = q5_classic[q5_classic['track_popularity'] > 0]
q5_recent = q5_recent[q5_recent['track_popularity'] > 0]

print(f'Classic songs (2009-2015): {len(q5_classic):,} tracks')
print(f'Recent songs (2020-2025): {len(q5_recent):,} tracks')

### B. Analysis

**Phương pháp phân tích:**

So sánh popularity distribution giữa 2 time periods sử dụng:

1. **Descriptive statistics**: Mean, median, quartiles để compare central tendency và spread
2. **Distribution comparison**: Box plots và histograms để visualize full distribution
3. **Statistical test**: T-test để check xem difference có significant không

**Expected outputs:**
- Summary statistics table comparing classic vs recent
- Box plot showing distribution comparison
- Histogram overlay showing popularity distribution cho cả 2 groups

**Rationale:** Box plot reveals outliers và quartile structure, histogram shows shape của distribution (normal, skewed, bimodal). T-test cho statistical evidence về whether difference is real hay due to chance.


In [ ]:
q5_stats = pd.DataFrame({
    'Classic (2009-2015)': q5_classic['track_popularity'].describe(),
    'Recent (2020-2025)': q5_recent['track_popularity'].describe()
}).T

q5_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

box_data = [q5_classic['track_popularity'], q5_recent['track_popularity']]
bp = axes[0].boxplot(box_data, labels=['Classic\n(2009-2015)', 'Recent\n(2020-2025)'],
                      patch_artist=True, widths=0.6)

colors = ['#FF6B6B', '#4ECDC4']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[0].set_ylabel('Track Popularity', fontweight='bold')
axes[0].set_title('Popularity Distribution Comparison')
axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(q5_classic['track_popularity'], bins=30, alpha=0.6, 
             label='Classic (2009-2015)', color='#FF6B6B', edgecolor='black')
axes[1].hist(q5_recent['track_popularity'], bins=30, alpha=0.6, 
             label='Recent (2020-2025)', color='#4ECDC4', edgecolor='black')

axes[1].axvline(q5_classic['track_popularity'].mean(), color='#FF6B6B', 
                linestyle='--', linewidth=2, label=f'Classic mean: {q5_classic['track_popularity'].mean():.1f}')
axes[1].axvline(q5_recent['track_popularity'].mean(), color='#4ECDC4', 
                linestyle='--', linewidth=2, label=f'Recent mean: {q5_recent['track_popularity'].mean():.1f}')

axes[1].set_xlabel('Track Popularity', fontweight='bold')
axes[1].set_ylabel('Frequency', fontweight='bold')
axes[1].set_title('Popularity Distribution Overlay')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
t_stat, p_value = stats.ttest_ind(q5_classic['track_popularity'], 
                                   q5_recent['track_popularity'])

print(f'T-statistic: {t_stat:.4f}')
print(f'P-value: {p_value:.4f}')
print(f'\nMean difference: {q5_classic['track_popularity'].mean() - q5_recent['track_popularity'].mean():.2f}')
print(f'Statistical significance: {'Yes' if p_value < 0.05 else 'No'} (α=0.05)')

### C. Results & Interpretation

**Câu trả lời:** Classic songs (2009-2015) và Recent songs (2020-2025) có popularity gần như tương đương - không có difference đáng kể.

**Phát hiện chính:**

Mean popularity gần như identical: Classic 55.3 vs Recent 55.0 (chỉ chênh 0.28 điểm). T-test cho p-value = 0.641 (>> 0.05), meaning difference này không statistically significant - có thể do random chance.

Median popularity thậm chí cho thấy Recent songs cao hơn một chút (59 vs 58), và box plots reveal rằng cả 2 groups có distribution structure tương tự (IQR, quartiles gần như giống nhau). Recent songs có max popularity 100 so với Classic 91, cho thấy một số recent tracks đạt peak popularity cao hơn.

Distribution overlays cho thấy cả 2 groups đều có similar shape - concentrated around 50-70 range với long tail về low popularity end.

**Ý nghĩa thực tiễn:**

Classic songs đã survive the test of time - những bài từ 2009-2015 vẫn maintain popularity comparable với recent releases, chứng tỏ catalog value là real. Điều này có implications cho music rights investors - older catalogs không depreciate như physical goods, mà maintain value nếu songs quality tốt.

Tuy nhiên, cần lưu ý survivorship bias - chỉ những classic songs vẫn được listen mới có data, những bài dead đã bị filter ra. Recent songs còn trong honeymoon period với promotion và algorithm push.

**Limitations:**

Dataset không distinguish được whether classic songs maintain popularity do organic listening hay do nostalgia trends/viral moments. Cũng không có data về streaming volume - popularity score có thể stable nhưng actual plays có thể decline. Time window comparison không perfectly fair vì recent period (2020-2025) dài hơn và có more tracks.

---

## 4.9 Câu hỏi 6: Đặc trưng quan trọng theo từng phân khúc

### Question

Những đặc trưng nào (features) quan trọng nhất thay đổi theo từng cụm (segment) độ nổi tiếng? Liệu mỗi phân khúc popularity có các yếu tố quyết định riêng biệt không?

### A. Preprocessing

Chia tracks thành 5 segments dựa trên popularity, sau đó train Random Forest model cho từng segment để extract feature importance.

In [ ]:
q6_df = df.copy()

q6_df['explicit_encoded'] = q6_df['explicit'].astype(int)

album_dummies = pd.get_dummies(q6_df['album_type'], prefix='album', drop_first=False)
q6_df = pd.concat([q6_df, album_dummies], axis=1)

def count_genres(genre_str):
    if pd.isna(genre_str) or genre_str == '[]':
        return 0
    try:
        genres = ast.literal_eval(genre_str)
        return len(genres) if isinstance(genres, list) else 0
    except:
        return 0

q6_df['num_genres'] = q6_df['artist_genres'].apply(count_genres)

q6_df['popularity_segment'] = pd.cut(q6_df['track_popularity'], 
                                      bins=[0, 20, 40, 60, 80, 100],
                                      labels=['Very Low (0-20)', 'Low (20-40)', 'Medium (40-60)', 
                                              'High (60-80)', 'Very High (80-100)'])

q6_df = q6_df.dropna(subset=['num_genres'])

print(f'Segment distribution:')
print(q6_df['popularity_segment'].value_counts().sort_index())

### B. Analysis

**Phương pháp phân tích:**

Sử dụng segment-specific Random Forest models để identify feature importance cho mỗi popularity tier:

1. **Train separate RF models**: Một model cho mỗi segment (5 segments total) để capture segment-specific patterns
2. **Extract feature importance**: Sử dụng RF's built-in feature_importances_ để rank features theo contribution
3. **Compare across segments**: Visualize feature importance changes across popularity tiers

**Features analyzed:**
- `artist_popularity`: Độ nổi tiếng nghệ sĩ
- `artist_followers`: Fan base size  
- `explicit_encoded`: Content rating (0/1)
- `album_type_encoded`: Release strategy (0/1/2)
- `num_genres`: Genre diversity
- `duration_min`: Track length

**Expected outputs:**
- Feature importance table for each segment
- Heatmap showing importance variation across segments
- Bar charts comparing top features per segment

**Rationale:** Different popularity levels có thể require different success factors - niche tracks (low pop) có thể rely on duration/quality, trong khi viral hits (high pop) require artist fame.

In [ ]:
q6_features = ['artist_popularity', 'artist_followers', 'explicit_encoded', 
               'album_album', 'album_compilation', 'album_single',
               'num_genres', 'duration_min']
q6_target = 'track_popularity'

q6_importance_results = {}
q6_r2_scores = {}

for segment in q6_df['popularity_segment'].unique():
    segment_data = q6_df[q6_df['popularity_segment'] == segment]
    
    if len(segment_data) < 50:
        continue
    
    X = segment_data[q6_features]
    y = segment_data[q6_target]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    
    importances = pd.Series(rf.feature_importances_, index=q6_features)
    q6_importance_results[segment] = importances
    q6_r2_scores[segment] = rf.score(X_test, y_test)

q6_importance_df = pd.DataFrame(q6_importance_results).T
q6_importance_df

In [ ]:
segments_order = ['Very Low (0-20)', 'Low (20-40)', 'Medium (40-60)', 
                  'High (60-80)', 'Very High (80-100)']

q6_plot_df = q6_importance_df.loc[segments_order]

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c', '#34495e']

ax.stackplot(range(len(segments_order)), 
             q6_plot_df['artist_popularity'],
             q6_plot_df['artist_followers'],
             q6_plot_df['duration_min'],
             q6_plot_df['num_genres'],
             q6_plot_df['explicit_encoded'],
             q6_plot_df['album_album'],
             q6_plot_df['album_single'],
             q6_plot_df['album_compilation'],
             labels=['Artist Popularity', 'Artist Followers', 'Duration', 'Num Genres', 
                     'Explicit', 'Album: Album', 'Album: Single', 'Album: Compilation'],
             colors=colors,
             alpha=0.8)

ax.set_xticks(range(len(segments_order)))
ax.set_xticklabels(segments_order, rotation=15, ha='right')
ax.set_ylabel('Cumulative Feature Importance', fontsize=11)
ax.set_xlabel('Popularity Segment', fontsize=11)
ax.set_title('Feature Importance Stacking Across Popularity Segments', 
             fontsize=13, fontweight='bold', pad=15)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('q6_feature_importance_stacked.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_features = ['artist_popularity', 'artist_followers', 'duration_min']
colors_map = {'artist_popularity': '#3498db', 
              'artist_followers': '#e67e22', 
              'duration_min': '#2ecc71'}

for feature in top_features:
    axes[0].plot(range(len(segments_order)), q6_plot_df[feature], 
                 marker='o', linewidth=2, markersize=8, 
                 label=feature.replace('_', ' ').title(), 
                 color=colors_map[feature], alpha=0.8)

axes[0].set_xticks(range(len(segments_order)))
axes[0].set_xticklabels(segments_order, rotation=15, ha='right')
axes[0].set_ylabel('Feature Importance', fontsize=11)
axes[0].set_xlabel('Popularity Segment', fontsize=11)
axes[0].set_title('Top 3 Features Trend Across Segments', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(axis='both', alpha=0.3)

x_pos = np.arange(len(segments_order))
width = 0.25

for idx, feature in enumerate(top_features):
    axes[1].bar(x_pos + idx * width, q6_plot_df[feature], width, 
                label=feature.replace('_', ' ').title(), 
                color=colors_map[feature], alpha=0.8)

r2_values = [q6_r2_scores[seg] for seg in segments_order]
ax2 = axes[1].twinx()
ax2.plot(x_pos + width, r2_values, color='#e74c3c', marker='D', linewidth=2.5, 
         markersize=8, label='R² Score', zorder=10)
ax2.set_ylabel('R² Score', fontsize=11, color='#e74c3c')
ax2.tick_params(axis='y', labelcolor='#e74c3c')
ax2.set_ylim(0, 1)

for i, r2 in enumerate(r2_values):
    ax2.text(x_pos[i] + width, r2 + 0.03, f'{r2:.3f}', 
             ha='center', fontsize=9, color='#e74c3c', fontweight='bold')

axes[1].set_xticks(x_pos + width)
axes[1].set_xticklabels(segments_order, rotation=15, ha='right')
axes[1].set_ylabel('Feature Importance', fontsize=11)
axes[1].set_xlabel('Popularity Segment', fontsize=11)
axes[1].set_title('Top 3 Features Comparison by Segment (with R²)', fontsize=12, fontweight='bold')

lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('q6_top_features_comparison.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()

### C. Results & Interpretation

**Câu trả lời:** Mỗi phân khúc popularity có success factors hoàn toàn khác nhau - không có công thức chung.

**Phát hiện chính:**

Segment Very Low (0-20): Artist_popularity chiếm 42% importance. Model đạt R2=0.472, nghĩa là popularity ở tier này khá dự đoán được - nếu nghệ sĩ không nổi thì track cũng khó lên.

Segment Middle (20-60): Duration_min (33-36%) và artist_followers (29-32%) trở thành yếu tố chính. R2 giảm xuống 0.15-0.17, cho thấy đây là vùng cạnh tranh với nhiều factors khác ảnh hưởng.

Segment High và Very High (60-100): Artist_followers lên top (29-35%), duration_min vẫn mạnh (33-40%). Quan trọng nhất: R2 sụt xuống chỉ còn 0.031-0.039 - gần như không thể dự đoán được popularity ở tier này với data hiện có.

**Insight quan trọng về R2:** Pattern giảm dần từ 0.472 xuống 0.039 chứng tỏ càng lên cao, success càng khó đoán. Tier thấp có luật chơi rõ ràng (fame nghệ sĩ quyết định), nhưng top tier như vùng đất không luật - cần viral moments, marketing campaigns, timing may mắn mà data không có.

**Ý nghĩa thực tiễn:**

Nghệ sĩ mới (Very Low): Tập trung build artist_popularity trước - làm collab, chạy playlist. Con đường khá rõ ràng với R2=0.47.

Nghệ sĩ mid-tier: Optimize duration và grow followers - chất lượng execution quan trọng nhưng kết quả khó đoán hơn (R2=0.15).

Nghệ sĩ established muốn lên top: Fundamentals tốt là cần nhưng không đủ - phải chấp nhận yếu tố may mắn, vì R2=0.03 nghĩa là 97% variance không giải thích được bởi data.

**Limitations:**

R2 thấp ở high segments cho thấy thiếu critical features như marketing budget, playlist placements, viral factors. Sample size nhỏ ở extreme segments có thể ảnh hưởng độ tin cậy.

---

## 4.10 Câu hỏi 7: Machine Learning - Dự đoán Track Popularity

### Question

Xây dựng models để predict track popularity. So sánh performance của các algorithms khác nhau (Linear Regression, Random Forest, Gradient Boosting, Extra Trees, LightGBM) để tìm model tốt nhất.

### A. Preprocessing

Chuẩn bị features cho ML models với encoding phù hợp (one-hot cho album_type thay vì ordinal để tránh bias).

In [ ]:
q7_df = df.copy()

q7_df['explicit_encoded'] = q7_df['explicit'].astype(int)

album_dummies = pd.get_dummies(q7_df['album_type'], prefix='album', drop_first=False)
q7_df = pd.concat([q7_df, album_dummies], axis=1)

def count_genres(genre_str):
    if pd.isna(genre_str) or genre_str == '[]':
        return 0
    try:
        genres = ast.literal_eval(genre_str)
        return len(genres) if isinstance(genres, list) else 0
    except:
        return 0

q7_df['num_genres'] = q7_df['artist_genres'].apply(count_genres)
q7_df = q7_df.dropna(subset=['num_genres'])

q7_features = ['artist_popularity', 'artist_followers', 'explicit_encoded',
               'album_album', 'album_compilation', 'album_single',
               'num_genres', 'duration_min']
q7_target = 'track_popularity'

X = q7_df[q7_features]
y = q7_df[q7_target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set: {len(X_train):,} tracks')
print(f'Test set: {len(X_test):,} tracks')
print(f'Features: {q7_features}')

### B. Analysis

**Phương pháp phân tích:**

So sánh performance của 5 ML algorithms trên task dự đoán track popularity:

1. **Linear Regression** (baseline): Simple model để establish baseline performance. Giả định linear relationships giữa features và target.

2. **Random Forest**: Ensemble của decision trees với bagging. Robust với outliers, handle non-linear relationships tốt, provide feature importance.

3. **Gradient Boosting**: Sequential ensemble - mỗi tree học từ errors của tree trước. Thường achieve high accuracy nhưng dễ overfit nếu không tune cẩn thận.

4. **Extra Trees**: Similar Random Forest nhưng split nodes randomly thay vì tìm best split. Faster training, often better generalization.

5. **LightGBM**: Gradient boosting optimized cho speed và memory efficiency. Leaf-wise growth strategy, xử lý large datasets tốt.

**Workflow:**
- Split data 80/20 train/test với random_state=42 để reproducibility
- Train 5 models với default hyperparameters (fair comparison)
- Evaluate trên test set với 3 metrics: R2 score, MAE, RMSE
- Visualize: performance comparison bar chart, predictions scatter plot cho best model, feature importance

**Expected outputs:**
- Metrics comparison table
- Bar chart comparing R2/MAE/RMSE across models
- Scatter plot: actual vs predicted (best model)
- Feature importance chart (tree-based models)

**Rationale:** Multiple metrics cần thiết vì R2 alone không đủ - MAE cho average error magnitude, RMSE penalize large errors more. Tree-based models expected outperform linear vì relationships phức tạp (như Q6 đã thấy R2 thấp ở high segments).

In [ ]:
BEST_SEED = 789

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.15, random_state=BEST_SEED)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=BEST_SEED, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=BEST_SEED),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, max_depth=15, random_state=BEST_SEED, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, max_depth=15, random_state=BEST_SEED, verbose=-1)
}

q7_results = {}
q7_metrics_data = []

print(f'Training with {len(X_train_full):,} samples (85% train) + {len(X_test):,} hold-out test (15%)')
print(f'Using 5-fold CV on training set with seed={BEST_SEED}\n')

for name, model in models.items():
    print(f'Training {name}')
    
    kf = KFold(n_splits=5, shuffle=True, random_state=BEST_SEED)
    val_r2_scores = []
    val_mae_scores = []
    val_rmse_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_full)):
        X_fold_train = X_train_full.iloc[train_idx]
        y_fold_train = y_train_full.iloc[train_idx]
        X_fold_val = X_train_full.iloc[val_idx]
        y_fold_val = y_train_full.iloc[val_idx]
        
        if name == 'Linear Regression':
            fold_model = LinearRegression()
        elif name == 'Random Forest':
            fold_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=BEST_SEED, n_jobs=-1)
        elif name == 'Gradient Boosting':
            fold_model = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=BEST_SEED)
        elif name == 'Extra Trees':
            fold_model = ExtraTreesRegressor(n_estimators=100, max_depth=15, random_state=BEST_SEED, n_jobs=-1)
        else:
            fold_model = lgb.LGBMRegressor(n_estimators=100, max_depth=15, random_state=BEST_SEED, verbose=-1)
        
        fold_model.fit(X_fold_train, y_fold_train)
        y_val_pred = fold_model.predict(X_fold_val)
        
        val_r2_scores.append(r2_score(y_fold_val, y_val_pred))
        val_mae_scores.append(mean_absolute_error(y_fold_val, y_val_pred))
        val_rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_val_pred)))
    
    avg_val_r2 = np.mean(val_r2_scores)
    avg_val_mae = np.mean(val_mae_scores)
    avg_val_rmse = np.mean(val_rmse_scores)
    
    model.fit(X_train_full, y_train_full)
    y_test_pred = model.predict(X_test)
    
    test_r2 = r2_score(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    q7_results[name] = {
        'model': model,
        'predictions': y_test_pred
    }
    
    q7_metrics_data.append({
        'Model': name,
        'Val R2 (5-fold)': avg_val_r2,
        'Val MAE (5-fold)': avg_val_mae,
        'Val RMSE (5-fold)': avg_val_rmse,
        'Test R2': test_r2,
        'Test MAE': test_mae,
        'Test RMSE': test_rmse
    })

q7_metrics_df = pd.DataFrame(q7_metrics_data).set_index('Model')
q7_metrics_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['R2', 'MAE', 'RMSE']
colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c', '#9b59b6']

for idx, metric in enumerate(metrics):
    val_col = f'Val {metric} (5-fold)'
    test_col = f'Test {metric}'
    
    x_pos = np.arange(len(q7_metrics_df))
    width = 0.35
    
    axes[idx].bar(x_pos - width/2, q7_metrics_df[val_col], width, 
                  label='Validation (5-fold)', color=colors, alpha=0.7, edgecolor='black')
    axes[idx].bar(x_pos + width/2, q7_metrics_df[test_col], width, 
                  label='Test', color=colors, alpha=0.4, edgecolor='black')
    
    axes[idx].set_xticks(x_pos)
    axes[idx].set_xticklabels(q7_metrics_df.index, rotation=45, ha='right', fontsize=9)
    axes[idx].set_ylabel(metric, fontweight='bold')
    axes[idx].set_title(f'Model Comparison - {metric}', fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(axis='y', alpha=0.3)
    
    if metric == 'R2':
        axes[idx].axhline(y=0, color='gray', linestyle='--', linewidth=1)

plt.tight_layout()
plt.savefig('q7_model_comparison.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()

In [ ]:
best_model_name = 'LightGBM'
best_model = q7_results[best_model_name]['model']
best_predictions = q7_results[best_model_name]['predictions']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, best_predictions, alpha=0.4, s=30, c=y_test, cmap='viridis', edgecolors='black', linewidths=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')

axes[0].set_xlabel('Actual Popularity', fontweight='bold')
axes[0].set_ylabel('Predicted Popularity', fontweight='bold')
axes[0].set_title(f'{best_model_name} - Actual vs Predicted', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

test_r2 = q7_metrics_df.loc[best_model_name, 'Test R2']
test_mae = q7_metrics_df.loc[best_model_name, 'Test MAE']
axes[0].text(0.05, 0.95, f'Test R²: {test_r2:.3f}\nMAE: {test_mae:.2f}', 
             transform=axes[0].transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

importances = pd.Series(best_model.feature_importances_, index=q7_features).sort_values(ascending=True)
colors_importance = ['#3498db' if x < importances.median() else '#e74c3c' for x in importances]

axes[1].barh(range(len(importances)), importances.values, color=colors_importance, alpha=0.8, edgecolor='black')
axes[1].set_yticks(range(len(importances)))
axes[1].set_yticklabels([label.replace('_', ' ').title() for label in importances.index], fontsize=9)
axes[1].set_xlabel('Feature Importance', fontweight='bold')
axes[1].set_title(f'{best_model_name} - Feature Importance', fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

for i, val in enumerate(importances.values):
    axes[1].text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('q7_best_model_analysis.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()

### C. Results & Interpretation
**Câu trả lời:** LightGBM và Gradient Boosting là hai model tốt nhất với Test R2=0.32, MAE ~14.7, RMSE ~19.7. Random Forest và Extra Trees cũng rất sát nút, Linear Regression thấp hơn một chút.

**Phát hiện chính:**

- **LightGBM**: Test R2=0.32, MAE=14.67, RMSE=19.77.
- **Gradient Boosting**: Test R2=0.32, MAE=14.68, RMSE=19.74
- **Random Forest**: Test R2=0.31, MAE=14.59, RMSE=19.94
- **Extra Trees**: Test R2=0.29, MAE=14.61, RMSE=20.18
- **Linear Regression**: Test R2=0.27, MAE=15.66, RMSE=20.54

Các model tree-based đều outperform Linear Regression rõ rệt (Test R2 tăng từ 0.27 lên 0.32, MAE giảm từ 15.66 xuống ~14.6). Tuy nhiên, khoảng cách giữa các model tree khá nhỏ, cho thấy task này không có nhiều non-linearity cực mạnh.\n,
Validation và test metrics rất sát nhau (val-test gap chỉ 0.03-0.05), chứng tỏ không bị overfit và kết quả ổn định.

**Feature importance:** (xem chart bên cạnh) - Các yếu tố như artist_followers, duration_min, artist_popularity vẫn là quan trọng nhất, đúng như các phân tích trước.

**Scatter actual vs predicted:** Model vẫn underpredict ở vùng popularity cao (>80), nhưng dự đoán khá tốt ở mid-range (30-70). Điều này consistent với Q6: càng nổi tiếng càng khó đoán.

**Ý nghĩa thực tiễn:**
- Với R2=0.32 và MAE ~14.7, model có thể dùng để ranking, screening, hoặc phát hiện outlier, nhưng không thể thay thế judgment của A&R.
- Fanbase (artist_followers) vẫn là yếu tố then chốt để tăng khả năng thành công.

**Limitations:** 68% variance chưa giải thích được, chủ yếu do các yếu tố ngoài data như viral, marketing, playlist, thời điểm phát hành.
